# Aula 6 — Identificação de UCEs com PHYLUCE

Nesta etapa começamos a usar o ambiente específico do **PHYLUCE**.

O PHYLUCE possui uma cadeia de dependências extensa; por isso, não será misturado ao ambiente
geral `bioinfo`. Utilizaremos o ambiente oficial `phyluce-1.7.3`.

## 0. Preparar o runtime

O Google Drive guarda os **dados** entre as aulas, mas o runtime do Colab é temporário.
Programas instalados no runtime podem desaparecer quando a sessão termina.

Por isso, quando uma aula precisar de ferramentas externas, começaremos verificando se
o Conda já existe. Se não existir, ele será instalado antes de qualquer outra configuração.

> Esta deve ser a primeira célula executável do notebook, porque a instalação do Conda
> pode reiniciar o runtime.

In [ ]:
import shutil

if shutil.which("conda"):
    print("Conda já está disponível neste runtime.")
else:
    !pip install -q condacolab
    import condacolab
    condacolab.install()

### Verificar o Conda e configurar Bioconda

Usaremos a configuração recomendada pelo Bioconda: `conda-forge` com maior prioridade,
seguido de `bioconda`, e prioridade estrita.

Como `conda config --add` adiciona canais do menor para o maior nível de prioridade,
executamos primeiro `bioconda` e depois `conda-forge`.

In [ ]:
!conda --version
!conda config --remove-key channels 2>/dev/null || true
!conda config --add channels bioconda
!conda config --add channels conda-forge
!conda config --set channel_priority strict
!conda config --show channels

## 1. Retomar o projeto no Google Drive

Todas as práticas usam a mesma raiz:

`/content/drive/MyDrive/Bioinformatica_Biologia_Molecular`

Os resultados de uma aula são lidos pela aula seguinte. Assim, os **dados persistem**
mesmo quando o runtime do Colab é encerrado.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RUN = "SRR15736591"
SAMPLE = "hypochilus_petrunkevitchi_SRR15736591"

PASTAS = {
    "01_bancos": ROOT / "01_bancos",
    "02_blast": ROOT / "02_blast",
    "03_raw": ROOT / "03_sra_fastq" / "raw-fastq",
    "04_qc": ROOT / "04_qc_trimming",
    "04_trimmed": ROOT / "04_qc_trimming" / "trimmed",
    "05_assemblies": ROOT / "05_spades" / "spades-assemblies",
    "05_contigs": ROOT / "05_spades" / "spades-assemblies" / "contigs",
    "06_match": ROOT / "06_uce_match",
    "06_probes": ROOT / "06_uce_match" / "probes",
    "06_results": ROOT / "06_uce_match" / "uce-search-results",
    "07_taxon_sets": ROOT / "07_uce_extract" / "taxon-sets" / "all",
    "08_integracao": ROOT / "08_integracao",
    "ambientes": ROOT / "ambientes",
}

for pasta in PASTAS.values():
    pasta.mkdir(parents=True, exist_ok=True)

os.chdir(ROOT)

print("Diretório atual:", Path.cwd())
print("\nEstrutura principal do projeto:")
for chave, pasta in PASTAS.items():
    print(f"{chave:15s} -> {pasta.relative_to(ROOT)}")

## 2. Verificar os contigs produzidos na Aula 5

In [ ]:
CONTIGS_DIR = PASTAS["05_contigs"]
CONTIG_FILE = CONTIGS_DIR / f"{SAMPLE}.contigs.fasta"

PROBES_DIR = PASTAS["06_probes"]
RESULTS = PASTAS["06_results"]
MATCH_ROOT = PASTAS["06_match"]

if not CONTIG_FILE.exists():
    raise FileNotFoundError(
        "Contigs preparados para PHYLUCE não encontrados. Execute primeiro a Aula 5."
    )

print("Entrada:", CONTIG_FILE)
print("Diretório de contigs:", CONTIGS_DIR)
print("Saída dos matches:", RESULTS)

## 3. Criar o ambiente oficial do PHYLUCE

A documentação do PHYLUCE 1.7.3 recomenda criar o ambiente a partir do arquivo YAML
oficial da versão para Linux. Salvaremos esse YAML no Drive para registrar a instalação.

In [ ]:
PHY_ENV = "phyluce-1.7.3"
PHY_YML = PASTAS["ambientes"] / "phyluce-1.7.3-py36-Linux-conda.yml"
PHY_URL = "https://raw.githubusercontent.com/faircloth-lab/phyluce/v1.7.3/distrib/phyluce-1.7.3-py36-Linux-conda.yml"

if not PHY_YML.exists():
    import urllib.request
    urllib.request.urlretrieve(PHY_URL, PHY_YML)

print("YAML:", PHY_YML)

envs = !conda env list
if not any(line.split() and line.split()[0] == PHY_ENV for line in envs if not line.startswith("#")):
    !conda env create -n "$PHY_ENV" --file "$PHY_YML"
else:
    print("Ambiente PHYLUCE já existe neste runtime.")

## 4. Verificar o PHYLUCE

In [ ]:
!conda run -n "$PHY_ENV" phyluce_assembly_match_contigs_to_probes --help | head -20

## 5. Obter o conjunto de sondas Arachnida 1.1Kv1

O site ultraconserved.org lista o conjunto Arachnida 1.1Kv1 com 14.799 baits para 1.120 UCEs.
O arquivo oficial é distribuído por Figshare.

In [ ]:
import urllib.request, zipfile, tarfile, shutil

download = PROBES_DIR / "arachnida_1.1Kv1_download"
url = "https://ndownloader.figshare.com/files/6042078"

if not download.exists():
    urllib.request.urlretrieve(url, download)

extract_dir = PROBES_DIR / "extracted"
extract_dir.mkdir(exist_ok=True)

try:
    with zipfile.ZipFile(download) as z:
        z.extractall(extract_dir)
except Exception:
    pass

try:
    with tarfile.open(download) as t:
        t.extractall(extract_dir)
except Exception:
    pass

candidates = []
for pattern in ["*.fasta", "*.fa", "*.fas", "*.fna"]:
    candidates.extend(extract_dir.rglob(pattern))

if candidates:
    PROBES_FASTA = max(candidates, key=lambda p: p.stat().st_size)
else:
    # alguns downloads são o próprio FASTA sem extensão informativa
    if download.read_text(errors="ignore").lstrip().startswith(">"):
        PROBES_FASTA = PROBES_DIR / "arachnida_1.1Kv1.fasta"
        shutil.copy2(download, PROBES_FASTA)
    else:
        raise FileNotFoundError(
            "O download foi obtido, mas o FASTA de probes não foi localizado automaticamente."
        )

print("Probes:", PROBES_FASTA)

## 6. Comparar contigs com as sondas

In [ ]:
import shutil

if RESULTS.exists():
    shutil.rmtree(RESULTS)

!conda run -n "$PHY_ENV" phyluce_assembly_match_contigs_to_probes   --contigs "$CONTIGS_DIR"   --probes "$PROBES_FASTA"   --output "$RESULTS"   --min-coverage 80   --min-identity 80

## 7. Criar o conjunto de táxons

In [ ]:
TAXON_CONF = MATCH_ROOT / "taxon-set.conf"
TAXON_CONF.write_text(f"[all]\n{SAMPLE}\n")
print(TAXON_CONF.read_text())

## 8. Verificar o banco de matches

In [ ]:
DB = RESULTS / "probe.matches.sqlite"

if not DB.exists():
    raise FileNotFoundError("probe.matches.sqlite não foi produzido.")

print("Banco de matches:", DB)
!ls -lh "$RESULTS"

## Saída para a próxima aula

A Aula 7 usará:
- `06_uce_match/uce-search-results/probe.matches.sqlite`
- `06_uce_match/taxon-set.conf`
- o diretório de contigs da Aula 5.

Assim, identificação e extração ficam separadas em duas práticas conectadas.